<a href="https://colab.research.google.com/github/mille-s/OpenReview_API/blob/main/OpenReviewAPI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title Setup

! pip install openreview-py
import openreview
from openreview import api

my_email = ''#@param{type:"string"}
my_password = ''#@param{type:"string"}

# API V2
client = api.OpenReviewClient(
    baseurl = 'https://api2.openreview.net',
    username = my_email,
    password = my_password
)

groups = client.get_groups(member=my_email)
chair_groups = [g.id for g in groups if 'Chair' in g.id or 'Program_Chairs' in g.id]
print("You're chair of:", chair_groups)

In [ ]:
#@title Get Invitations (Choose venue ID and run cell)
# This is apparently the URLs where we can find info. Used that for the cell above, to get the Decision. But couldn't find a way to access decisions in the end.

# venue_id = 'aclweb.org/ACL/2025/Workshop/GEM'
# venue_id = 'aclweb.org/ACL/2026/Workshop/GEM
venue_id = 'aclweb.org/ACL/2026/Workshop/GEM'#@param['aclweb.org/ACL/2025/Workshop/GEM', 'aclweb.org/ACL/2026/Workshop/GEM', 'aclweb.org/ACL/2026/Workshop/GEM_ARR_Commitment']
invitations = client.get_invitations(prefix=venue_id)

print("Invitations:")
for inv in invitations:
    print(inv.id)

## Deploy assignments

In [ ]:
#@title Extract info submissions
import csv

submissions = client.get_all_notes(
    invitation=venue_id+'/-/Submission'
)
print(len(submissions))

rows = []
dico_forumID_subNum = {}
for n in submissions:
    track = n.content.get('track') or n.content.get('Track')
    # track might be a dict with a 'value', depending on how GEM configured the form
    id_review = n.content.get('open_review_id_of_reviewing_authors')['value']
    if isinstance(track, dict) and 'value' in track:
        track = track['value']

    rows.append({
        'id': n.id,
        'number': n.number,
        'title': n.content.get('title')['value'],
        'track': track,
        'reviewer_ID': id_review
    })
    dico_forumID_subNum[n.id] = n.number

with open('gem_tracks.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['id', 'number', 'title', 'track', 'reviewer_ID'])
    writer.writeheader()
    writer.writerows(rows)

In [ ]:
#@title Check assignments via the API
# venue_id = 'aclweb.org/ACL/2026/Workshop/GEM'
group_of_interest = 'Area_Chairs'#@param['Reviewers', 'Area_Chairs']
reviewer_group_id = f'{venue_id}/'+group_of_interest
assignment_inv = f'{reviewer_group_id}/-/Assignment'
all_assign = client.get_edges_count(invitation=assignment_inv)
if group_of_interest == 'Reviewers':
  print(f'Total assignments now: {all_assign} (expected: 237 + 32 = 269)')  # Expect ~269
  print('In 2026, after playing around with the code I ended up creating 32 other assignments and deleted one; these have not been deployed but still appear on this query (269+31=300).' )
elif group_of_interest == 'Area_Chairs':
  print(f'Total assignments now: {all_assign} (expected: 79 + 16 = 95)')  # Expect ~269

recent = client.get_all_edges(invitation=assignment_inv)
for e in recent[:5]:
    print(f'New: paper {e.head} -> reviewer {e.tail} (w={e.weight})')

print('Live API view: https://api2.openreview.net/edges?invitation=' + assignment_inv)

all_edges = client.get_all_edges(invitation=assignment_inv)
recent = all_edges[-10:]  # Last 10 (newest by default utime)
for e in recent:
    print(f'Paper {e.head} <- {e.tail} w={e.weight}')

In [ ]:
#@title Check new and old edges properties (it shows a mismatch regarding writers and signatures)
# New edges are edges that have been created via the UI (Assignment configuration console) but not deployed (because we cant deploy two assignment configurations via the UI, and I already deployed main papers).
# Perplexity's analysis: "Perfect diagnosis—the mismatch is clear! New edges lack Area Chairs (AC) in readers/writers and use Reviewers signature vs Program_Chairs (your PC group). PC console filters edges matching the official Reviewers/-/Assignment invitation config (includes ACs for oversight). UI-deployed main assignments match exactly; API ones don't."
new_edges = client.get_all_edges(invitation=f'{reviewer_group_id}/-/Assignment')[-1]  # Last one
print('New edge details:')
print(f'ID: {new_edges.id}')
print(f'Head (paper): {new_edges.head}')
print(f'Tail (reviewer): {new_edges.tail}')
print(f'Readers: {new_edges.readers}')
print(f'Writers: {new_edges.writers}')
print(f'Signatures: {new_edges.signatures}')
print(f'Label: {new_edges.label}')

old_edges = client.get_all_edges(invitation=f'{reviewer_group_id}/-/Assignment')[0]  # First one (I had 2 assignments)
print('\nOld edge details:')
print(f'ID: {old_edges.id}')
print(f'Head (paper): {old_edges.head}')
print(f'Tail (reviewer): {old_edges.tail}')
print(f'Readers: {old_edges.readers}')
print(f'Writers: {old_edges.writers}')
print(f'Signatures: {old_edges.signatures}')
print(f'Label: {old_edges.label}')

In [ ]:
#@title Get list of paper IDs that have the mismatch
# We simply check the latest assignments that haven't been deployed (those for non-archival papers).

# For reviewers, we had 32 non deployed assignments, for ACs 16
# number_assignments_not_deployed = 16 # @param {type:"slider", min:0, max:100, step:1}
# The assignment configuration names are in https://openreview.net/assignments?group=aclweb.org/ACL/2026/Workshop/GEM/Reviewers and https://openreview.net/assignments?group=aclweb.org/ACL/2026/Workshop/GEM/Area_Chairs
assignment_configuration_name = 'GEM2026_ACs_NonArchival'#@param['GEM26_final2_NonArchival', 'GEM2026_ACs_NonArchival']
all_edges = client.get_all_edges(invitation=f'{reviewer_group_id}/-/Assignment')

# Newest papers
# new_edges = all_edges[-number_assignments_not_deployed:]
# print(f'Number of papers you plan to deploy: {len(new_edges)}')

# 1. From original proposed (guarantees the papers in the last assignment configuration)
prop_nonarch = [e for e in client.get_all_edges(invitation=f'{reviewer_group_id}/-/Proposed_Assignment', label=assignment_configuration_name)]
print(f'Proposed in assignment configuration file: {len(prop_nonarch)}')  # If 0, there is nothing to be deployed
print('Is this the number you expected? If so, proceed to the next cell!')

if group_of_interest == 'Reviewers':
  # 2. Edges without AC readers (your API ones)
  no_ac_edges = [e for e in all_edges if 'Area_Chairs' not in str(e.readers)]
  print(f'No AC readers (API): {len(no_ac_edges)}')

  # 3. Full list paper IDs for no_ac_edges
  paper_IDs_no_ac_edges = list(set(e.head for e in no_ac_edges))
  print('Suspect papers:', paper_IDs_no_ac_edges)

In [ ]:
#@title Deploy assignments and update the metadata of the papers (IMPORTANT: READ COMMENTS AT THE TOP BEFORE RUNNING)

# In order to get no errors, there is an assignment config missing for non-archival papers.
# Like this one: https://openreview.net/invitation/edit?id=aclweb.org/ACL/2026/Workshop/GEM/Reviewers/-/Assignment
# Where "withContent": {"track": "Main"} should be "withContent": {"track": "Extended abstract (Non-archival)"}.
# I found no way to create a new assignment config, or to modify the existing one to accept both types of submissions.
# I ended up temporarily replacing "Main" by "Extended abstract (Non-archival)" in the file above, just the time to deploy the assignments.

# print(dico_forumID_subNum)
pc_group_id = f'{venue_id}/Program_Chairs'

for i, e in enumerate(prop_nonarch):
    # Get the submission ID that correponds to the forum ID. The submission ID is needed for the AC group ID
    # E.g. forumID = 9BKvW5dws9 , Submission ID = 42.
    subID = dico_forumID_subNum.get(e.head)
    ac_group_id = f'{venue_id}/Submission{str(subID)}/Area_Chairs'
    print(ac_group_id)
    readers_group = [venue_id, e.tail]
    writers_group = [venue_id]
    # Reviewers need ACs in readers and writers
    if group_of_interest == 'Reviewers':
      readers_group.append(ac_group_id)
      writers_group.append(ac_group_id)
    new_e_dict = {
        'invitation': f'{reviewer_group_id}/-/Assignment',
        'label': None,
        'weight': e.weight,
        'head': e.head,
        'tail': e.tail,
        'readers': readers_group,
        'writers': writers_group,
        'signatures': [pc_group_id]
    }
    new_e = openreview.Edge.from_json(new_e_dict)
    client.post_edge(new_e)
    print(f' {str(i)} - {e.head} -> {e.tail}')



In [ ]:
#@title Diagnosing state of different edges

# 1. Your main assignment invitation (the one the PC console uses)
ASSIGN_INV = f'{venue_id}/Reviewers/-/Assignment'

# 2. The invitation used for inviting reviewers
venue_group = client.get_group(venue_id)
INVITE_INV = venue_group.content['reviewers_invite_assignment_id']['value']

edges_assign = list(client.get_all_edges(invitation=ASSIGN_INV))
edges_invite = list(client.get_all_edges(invitation=INVITE_INV))

print(f'Edges in main assignment invitation {ASSIGN_INV}: {len(edges_assign)}')
print(f'Edges in invite assignment invitation {INVITE_INV}: {len(edges_invite)}')

# Build sets of (head, tail) for each
assign_keys = {(e.head, e.tail) for e in edges_assign}
invite_keys = {(e.head, e.tail) for e in edges_invite}

if assign_keys == invite_keys:
    print('✅ Both invitations have the same paper–reviewer pairs.')
else:
    print('🟡 The two invitations differ.')
    print('  Extra in ASSIGN:', len(assign_keys - invite_keys))
    print('  Extra in INVITE:', len(invite_keys - assign_keys))

## Post-Decision

In [ ]:
#@title Get list of IDs of accepted paper

# Handcompiled lists to use as backup
# List up to date as of 23/06/2025
# accepted_papers_2025 = [1, 3, 6, 8, 9, 11, 12, 15, 16, 17, 18, 19, 20, 22, 25, 26, 27, 28, 29, 30, 32, 33, 34, 36, 37, 39, 40, 41, 43, 45, 46, 47, 48, 50, 53, 54, 56, 57, 58, 59, 60, 61, 63, 64, 67, 68, 69, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 87, 88, 89, 91, 92, 93, 94, 95, 96, 100, 102, 103, 104, 105, 106, 108, 109, 113, 114, 115, 117]
# list_ids_accepted_submissions = []
# if 'ARR_Commitment' in venue_id:
#   list_ids_accepted_submissions = [2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 13, 14, 15]
# else:
#   list_ids_accepted_submissions = [2, 3, 6, 7, 9, 10, 11, 12, 14, 15, 17, 19, 20, 21, 22, 23, 24, 26, 28, 29, 30, 32, 33, 34, 35, 37, 38, 39, 40, 41, 42, 44, 46, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 63, 65, 66, 67, 68, 72, 73, 75, 76, 77, 81, 82, 83, 84, 86, 87, 89, 90, 91, 92, 94, 95, 96, 97, 98, 99]

accepted_submissions = client.get_all_notes(content={'venueid':venue_id} )
list_ids_accepted_submissions = sorted([n.number for n in accepted_submissions])
print(f'Accepted submissions: {len(accepted_submissions)}')
print(list_ids_accepted_submissions)
# print(accepted_submissions)

In [ ]:
#@title Get accepted papers author list with emails (Add IDs of accepted papers manually)
import re

print_authors = False #@param {type:"boolean"}
print_titles = False #@param {type:"boolean"}
# Query OR
venue_group = client.get_group(venue_id)
submission_name = venue_group.content['submission_name']['value']
# Get all submissions
all_submissions = client.get_all_notes(invitation=venue_id+f'/-/{submission_name}')
print(f"Found {len(all_submissions)} submissions.")

dico_submissions = {}

# submission = all_submissions[0]
# print(submission.content.keys())
# print(submission.content)
# print(submission.readers)
# print(submission.nonreaders)
# print(submission.writers)
# print(1/0)

# Organise accepted submissions by IDs
for submission in all_submissions:
  # Get submission ID
  # readers = submission.content['paperhash']['readers']
  readers = submission.readers
  sub_id_orig = None
  sub_id = None
  for reader in readers:
    if re.search('Submission', reader):
      sub_id_orig = reader.rsplit('/', 1)[0].rsplit('/', 1)[1].split('Submission')[1]
      sub_id = sub_id_orig.zfill(3)
  dico_submissions[sub_id] = {}
  # Get title
  sub_title = submission.content['title']['value']
  # Get author list
  sub_authors = submission.content['authors']['value']
  # Get author IDs list
  sub_author_IDs = submission.content['authorids']['value']
  # Get Abstract
  sub_abstract = submission.content['abstract']['value']
  # Get track
  # sub_track = submission.content['track']['value']
  # Get keywords
  sub_keywords = submission.content['keywords']['value']
  # Get Decision; doesn't work
  # sub_decisions = list(openreview.tools.iterget_notes(client, invitation=f'{GEM_ID}/Submission{sub_id}/-/Decision', forum=submission.forum))

  # Fill our dico with retrieved info
  dico_submissions[sub_id]['title'] = sub_title
  dico_submissions[sub_id]['authors'] = sub_authors
  dico_submissions[sub_id]['authorIDs'] = sub_author_IDs
  assert len(sub_authors) == len(sub_author_IDs)
  dico_submissions[sub_id]['abstract'] = sub_abstract
  # dico_submissions[sub_id]['track'] = sub_track
  dico_submissions[sub_id]['keywords'] = sub_keywords
  if sub_id in [str(x).zfill(3) for x in list_ids_accepted_submissions]:
    dico_submissions[sub_id]['decision'] = 'Accept'
  else:
    dico_submissions[sub_id]['decision'] = 'Reject'
  dico_submissions[sub_id]['authorEmails'] = []
  # Get email of authors; what I get here are anonymous emails despite being listed as chair...
  for author_id in sub_author_IDs:
    try:
      profile = client.get_profile(author_id)
      # The preferred email (if accessible)
      # emails = profile.content.get('emails', [])
      emails = profile.content['emails']
      dico_submissions[sub_id]['authorEmails'].append(emails[-1])
    except openreview.OpenReviewException as e:
      print(f"Error retrieving profile for {author_id}: {e}")
      dico_submissions[sub_id]['authorEmails'].append(author_id)
  # print(submission.content.get('email'))
  # print(submission.content.get('contact'))
  # print(submission.content.get('authorids'))


# Print Authors
count_accept = 0
for key, value in sorted(dico_submissions.items()):
  if dico_submissions[key]['decision'] == 'Accept':
    # print(key)
    string_sheet = ''
    string_sheet_title = dico_submissions[key]['title']
    # string_sheet_abstract = dico_submissions[key]['abstract']
    for i, author in enumerate(dico_submissions[key]['authors']):
      if re.search(' ', author):
        first_name = author.split(' ')[0]
        last_name = author.split(' ')[1]
      else:
        first_name = author
        last_name = ''
      # string_sheet += first_name + '\t' + last_name + '\t' + ' ' + '\t'
      # string_sheet += first_name + '\t' + last_name + '\t' + dico_submissions[key]['authorEmails'][i] + '\t'
      # Most emails are anonymised, so I use the ID instead
      string_sheet += first_name + '\t' + last_name + '\t' + dico_submissions[key]['authorIDs'][i] + '\t'
    if print_authors:
      print(string_sheet)
    if print_titles:
      print(string_sheet_title)
    # print(string_sheet_abstract)
    count_accept += 1

assert count_accept == len(list_ids_accepted_submissions)
print(f'Found {count_accept} accepted papers.')

In [ ]:
#@title Get list of submitted Camera-ready papers

print(f'Accepted submissions: {len(accepted_submissions)}')
accepted_archival_submissions = [note for note in accepted_submissions if not note.content['track']['value'] == 'Extended abstract (Non-archival)']
print(f'Acepted archival papers: {len(accepted_archival_submissions)}')

camera_ready_notes = []
for note in accepted_submissions:
    camera_ready_invitation = f'{venue_id}/Submission{note.number}/-/Camera_Ready_Revision'
    if camera_ready_invitation in note.invitations:
        camera_ready_notes.append(note)

print(f'Camera-ready papers submitted: {len(camera_ready_notes)}')
print(camera_ready_notes)